In [63]:
using LowLevelFEM, LinearAlgebra

In [64]:
openGeometry("boxes.geo")

In [65]:
#openPreProcessor()

In [66]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [67]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 2000)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

  0.426821 seconds (115.75 k allocations: 50.388 MiB, 1.35% gc time, 22.90% compilation time)


0

In [68]:
contact_pair = contact(u, master="master", slave="slave", cn=1e8, topology_tol=0.0)

Contact("slave" -> "master", 789 candidate nodes, 404 active, G=(2367, 9429), C=(2367, 2367))

In [69]:
using SparseArrays

# Lagrange multiplier field
Λ = Field([mat], type=:VectorField, dim=3, fieldName=:λ)

# Contact kinematics
L = contact(
    u,
    master="master",
    slave="slave",
    LagrangeMultiplierField=Λ,
    topology_tol=0.0,
    projection_tol=0.01
)

support = [bc_bottom, bc_top]
free_u = freeDoFs(U, support)

u_it = copy(u)
λ_it = vectorField(Λ, "body", [0, 0, 0])

# Zero multiplier block
nλ = size(L.E, 1)
Zλ = SystemMatrix(spzeros(nλ, nλ), Λ)

# Primal-dual active-set parameter.
# This is NOT a penalty stiffness; it is used only to determine the active set.
κ = 1e8

active_old = falses(length(L.slave_nodes))

for iter in 1:40

    # Current contact geometry
    updateContact!(L, u_it)

    (; G, g, E) = L

    pdim = L.U.pdim

    # Normal component in the reduced contact space
    normal_rows = 1:pdim:length(g)

    # Corresponding normal multiplier DoFs
    λn_dofs = L.multiplier_dofs[normal_rows]
    λn = DoFs(λ_it)[λn_dofs]

    # ----------------------------------------------------------
    # Primal-dual active set
    #
    # Sign convention:
    #     g_n >= 0       open/admissible
    #     λ_n <= 0       compression
    #
    # active <=> λ_n + κ g_n < 0
    # ----------------------------------------------------------
    active = λn .+ κ .* L.gap_values .< 0.0

    # Inactive multipliers are zero
    DoFs(λ_it)[λn_dofs[.!active]] .= 0.0

    # ----------------------------------------------------------
    # Active normal projector in contact space
    # ----------------------------------------------------------
    χ = zeros(Float64, length(g))
    χ[normal_rows[active]] .= 1.0

    P = SystemMatrix(
        spdiagm(0 => χ),
        nothing,
        nothing,
        nothing,
        nothing
    )

    # ----------------------------------------------------------
    # Contact constraint operator
    #
    #     B = E P G
    #     gλ = E P g
    # ----------------------------------------------------------
    B  = E * P * G
    gλ = E * P * g

    # ----------------------------------------------------------
    # KKT residual
    #
    #     rᵤ = K u - f + B' λ
    #     rλ = gλ
    # ----------------------------------------------------------
    rᵤ = K * u_it - f + B' * λ_it
    rλ = gλ

    # ----------------------------------------------------------
    # KKT tangent
    #
    #         [ K   B' ]
    #     A = [        ]
    #         [ B    0 ]
    # ----------------------------------------------------------
    A = SystemMatrix([
        K   B'
        B   Zλ
    ])

    r = SystemVector([rᵤ, rλ])

    # Offset of the multiplier field in the multifield system
    λoff = A.offsets[2]

    # Only displacement free DoFs and ACTIVE NORMAL multiplier DoFs
    # participate in the Newton correction.
    free = vcat(
        free_u,
        λoff .+ λn_dofs[active]
    )

    Δx = zeros(Float64, size(A, 1))

    Δx[free] =
        -A[free, free] \ r.a[free, 1]

    # Split the multifield correction
    Δu = @view Δx[1:λoff]
    Δλ = @view Δx[λoff+1:end]

    # Newton update
    DoFs(u_it)[:] .+= Δu
    DoFs(λ_it)[:] .+= Δλ

    # ----------------------------------------------------------
    # Convergence diagnostics
    # ----------------------------------------------------------
    Δactive = count(active .!= active_old)

    err_u =
        norm(Δu[free_u]) /
        max(norm(DoFs(u_it)[free_u]), eps())

    max_gap =
        any(active) ?
        maximum(abs.(L.gap_values[active])) :
        0.0

    λn = DoFs(λ_it)[λn_dofs]

    min_λ =
        any(active) ?
        minimum(λn[active]) :
        0.0

    println(
        "iter = ", iter,
        ", active = ", count(active),
        ", Δactive = ", Δactive,
        ", max |gap| = ", max_gap,
        ", min λn = ", min_λ,
        ", error = ", err_u
    )

    converged =
        Δactive == 0 &&
        max_gap < 1e-8 &&
        err_u < 1e-8

    active_old .= active

    converged && break
end

u_LM = u_it
λ_LM = λ_it

iter = 1, active = 404, Δactive = 404, max |gap| = 0.1764154831967565, min λn = -1724.0969079774677, error = 0.30993697318621105
iter = 2, active = 214, Δactive = 190, max |gap| = 6.90542040441783e-5, min λn = -3653.567886406623, error = 0.04590682400220624
iter = 3, active = 299, Δactive = 201, max |gap| = 0.05267661952906152, min λn = -2617.906707482418, error = 0.0481959413078862
iter = 4, active = 283, Δactive = 134, max |gap| = 0.033330005147686796, min λn = -2503.830586130748, error = 0.02950926272535046
iter = 5, active = 293, Δactive = 98, max |gap| = 0.03217498185955059, min λn = -2176.4689332797784, error = 0.024592478031101234
iter = 6, active = 296, Δactive = 63, max |gap| = 0.03226659510349194, min λn = -2361.172984892257, error = 0.019520969591988097
iter = 7, active = 304, Δactive = 48, max |gap| = 0.03008644751707676, min λn = -2189.7905790694526, error = 0.017733910802405477
iter = 8, active = 303, Δactive = 39, max |gap| = 0.02522409238856379, min λn = -2471.165926436

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [70]:
showDoFResults(u_LM, name="u cont.", visible=true, factor=1)


1

In [71]:
λ_LM = nodesToElements(λ_LM, onPhysicalGroup="slave")
showElementResults(λ_LM[1], name="λ")

2

In [72]:
λ_LM.type

:v3D

In [73]:
showElementResults(contact_pair.gap, name="gap")

3

In [74]:
openPostProcessor()

Két vagy több párnál majd:

```Julia
contacts = ContactSet(c1, c2, c3)

updateContact!(contacts, u_it)

Kc = sum(c.G' * c.C * c.G for c in contacts)
rc = sum(c.G' * c.C * c.g for c in contacts)

r = K * u_it - f + rc
A = K + Kc
```